## Example 1: Basic compression and decompression

In [ ]:
import lz4.block
import time

# Example 1: Basic compression and decompression
print("Example 1: Basic Compression and Decompression")

# Sample data to compress
original_data = b"This is a test string for LZ4 compression. " * 100
print(f"Original data size: {len(original_data)} bytes")
print(f"Sample content: {original_data[:50]}...")

# Compress the data
start_time = time.perf_counter()
compressed_data = lz4.block.compress(original_data)
compression_time = time.perf_counter() - start_time

print(f"Compressed size: {len(compressed_data)} bytes")
print(f"Compression ratio: {len(original_data) / len(compressed_data):.2f}:1")
print(f"Space saved: {100 * (1 - len(compressed_data) / len(original_data)):.1f}%")
print(f"Compression time: {compression_time * 1000:.2f} ms")

# Decompress the data
start_time = time.perf_counter()
decompressed_data = lz4.block.decompress(compressed_data)
decompression_time = time.perf_counter() - start_time

print(f"Decompressed size: {len(decompressed_data)} bytes")
print(f"Decompression time: {decompression_time * 1000:.2f} ms")

# Verify data integrity
assert decompressed_data == original_data
assert len(compressed_data) < len(original_data) // 10, len(compressed_data)  # highly repetitive input
print("Data integrity check: decompressed data matches the original")

# Show performance characteristics
total_time = compression_time + decompression_time
throughput_mb_s = (len(original_data) / (1024 * 1024)) / total_time

print(f"Total processing time: {total_time * 1000:.2f} ms")
print(f"Throughput: {throughput_mb_s:.1f} MB/s")

print("Basic compression completed successfully!")

## Example 2: Frame-based compression with streaming

In [ ]:
import lz4.frame
import io

# Example 2: Frame-based compression with streaming
print("Example 2: Frame-based Compression with Streaming")

# Create sample data with different patterns
patterns = [
    b"AAAAAAAAAAAAAAAAAAAAAAAAAA",  # Highly compressible
    b"The quick brown fox jumps over the lazy dog. " * 20,  # Text data
    b"1234567890" * 50,  # Numeric pattern
    bytes(range(256)) * 10,  # Binary data
]

combined_data = b"".join(patterns)
print(f"Combined data size: {len(combined_data)} bytes")

# Frame compression (includes headers and checksums)
print("\nFrame compression:")
compressed_frame = lz4.frame.compress(combined_data)

print(f"Frame compressed size: {len(compressed_frame)} bytes")
print(f"Frame compression ratio: {len(combined_data) / len(compressed_frame):.2f}:1")

# Decompress frame
decompressed_frame = lz4.frame.decompress(compressed_frame)
print(f"Frame decompression successful: {decompressed_frame == combined_data}")
assert decompressed_frame == combined_data

# Streaming compression
print(f"\nStreaming compression:")

# Compress in chunks
compressor = lz4.frame.LZ4FrameCompressor()
# begin() opens the frame and returns its header; a fresh compressor starts in the flushed state, so
# compress() without begin() raises "compress called after flush()".
compressed_chunks = [compressor.begin()]

chunk_size = 1000
for i in range(0, len(combined_data), chunk_size):
    chunk = combined_data[i:i + chunk_size]
    compressed_chunk = compressor.compress(chunk)
    if compressed_chunk:  # Only add non-empty chunks
        compressed_chunks.append(compressed_chunk)

# Finalize compression
final_chunk = compressor.flush()
if final_chunk:
    compressed_chunks.append(final_chunk)

# Combine compressed chunks
streaming_compressed = b"".join(compressed_chunks)
print(f"Streaming compressed size: {len(streaming_compressed)} bytes")

# Streaming decompression
decompressor = lz4.frame.LZ4FrameDecompressor()
decompressed_streaming = decompressor.decompress(streaming_compressed)

print(f"Streaming decompression successful: {decompressed_streaming == combined_data}")
assert decompressed_streaming == combined_data

# Compare compression methods
print(f"\nCompression method comparison:")
print(f"  Block compression: Simple, fast, no headers")
print(f"  Frame compression: Includes headers, checksums, metadata")
print(f"  Streaming compression: Process large data in chunks")

# Show frame benefits
frame_benefits = [
    "Built-in checksums: Data integrity verification",
    "Frame headers: Self-contained compressed data",
    "Streaming support: Process data without loading all into memory",
    "Interoperability: Compatible with LZ4 command-line tools",
    "Block linking: Improved compression for related blocks"
]

print(f"\nFrame compression benefits:")
for benefit in frame_benefits:
    print(f"  • {benefit}")

print("Frame-based compression completed successfully!")

## Example 3: Performance comparison and data type optimization

In [ ]:
import lz4.block
import json
import time

# Example 3: Performance comparison and data type optimization
print("Example 3: Performance Comparison and Data Type Optimization")

# Create different types of test data
test_datasets = {
    "text": "Lorem ipsum dolor sit amet, consectetur adipiscing elit. " * 200,
    "json": json.dumps({"users": [{"id": i, "name": f"user{i}", "active": i % 2 == 0} for i in range(100)]}),
    "repetitive": "ABCD" * 1000,
    "random": "".join([chr(65 + (i * 7) % 26) for i in range(4000)])
}

print("Performance comparison across data types:")
print(f"{'Data Type':<12} {'Original':<8} {'Compressed':<10} {'Ratio':<6} {'Time(ms)':<8}")
print("-" * 50)

results = {}

for name, data in test_datasets.items():
    data_bytes = data.encode('utf-8')
    original_size = len(data_bytes)
    
    # Measure compression performance
    start_time = time.time()
    compressed = lz4.block.compress(data_bytes)
    compression_time = (time.time() - start_time) * 1000
    
    compressed_size = len(compressed)
    ratio = original_size / compressed_size
    
    results[name] = {
        'original': original_size,
        'compressed': compressed_size,
        'ratio': ratio,
        'time': compression_time
    }
    
    print(f"{name:<12} {original_size:<8} {compressed_size:<10} {ratio:<6.1f} {compression_time:<8.2f}")
    
    # Verify decompression
    decompressed = lz4.block.decompress(compressed)
    assert decompressed == data_bytes, f"Decompression failed for {name}"

# Find best and worst compression
best_ratio = max(results.items(), key=lambda x: x[1]['ratio'])
worst_ratio = min(results.items(), key=lambda x: x[1]['ratio'])
fastest = min(results.items(), key=lambda x: x[1]['time'])

print(f"\nPerformance analysis:")
print(f"  Best compression ratio: {best_ratio[0]} ({best_ratio[1]['ratio']:.1f}:1)")
print(f"  Worst compression ratio: {worst_ratio[0]} ({worst_ratio[1]['ratio']:.1f}:1)")
print(f"  Fastest compression: {fastest[0]} ({fastest[1]['time']:.2f} ms)")

print("Performance comparison completed successfully!")

## Example 4: File compression and practical applications

In [ ]:
import lz4.block
import tempfile
import os

# Example 4: File compression and practical applications
print("Example 4: File Compression and Practical Applications")

# Create sample file data
file_contents = {
    "log_file": "2024-01-01 10:00:00 INFO Application started\n" * 500,
    "config_json": '{"database": {"host": "localhost", "port": 5432}, "cache": {"enabled": true, "ttl": 3600}}\n' * 200,
    "csv_data": "id,name,email,created_at\n" + "\n".join([f"{i},user{i},user{i}@example.com,2024-01-{i%30+1:02d}" for i in range(1000)])
}

print("File compression simulation:")

for file_type, content in file_contents.items():
    content_bytes = content.encode('utf-8')
    original_size = len(content_bytes)
    
    # Compress file content
    compressed = lz4.block.compress(content_bytes)
    compressed_size = len(compressed)
    
    # Calculate savings
    savings_bytes = original_size - compressed_size
    savings_percent = (savings_bytes / original_size) * 100
    
    print(f"\n{file_type.replace('_', ' ').title()}:")
    print(f"  Original size: {original_size:,} bytes")
    print(f"  Compressed size: {compressed_size:,} bytes")
    print(f"  Space saved: {savings_bytes:,} bytes ({savings_percent:.1f}%)")
    print(f"  Compression ratio: {original_size / compressed_size:.2f}:1")
    
    # Verify decompression
    decompressed = lz4.block.decompress(compressed)
    assert decompressed == content_bytes, file_type
    assert compressed_size < original_size, (file_type, compressed_size, original_size)
    print("  Integrity check: decompressed data matches the original")

# Practical use cases
print(f"\nPractical LZ4 applications:")

applications = [
    {
        "use_case": "Network transmission",
        "benefit": "Reduce bandwidth usage with fast compression",
        "scenario": "API responses, file transfers, streaming data"
    },
    {
        "use_case": "Database storage",
        "benefit": "Compress large text fields and JSON documents",
        "scenario": "Log tables, configuration storage, cached data"
    },
    {
        "use_case": "In-memory caching",
        "benefit": "Reduce memory footprint of cached objects",
        "scenario": "Redis compression, application caches, session data"
    },
    {
        "use_case": "File archival",
        "benefit": "Fast compression for backup and archival systems",
        "scenario": "Log rotation, data backups, temporary file compression"
    }
]

for app in applications:
    print(f"\n  {app['use_case']}:")
    print(f"    Benefit: {app['benefit']}")
    print(f"    Scenario: {app['scenario']}")

# Best practices
print(f"\nLZ4 best practices:")
best_practices = [
    "Use for speed-critical applications where decompression speed matters",
    "Consider block vs frame format based on your needs",
    "Test with your actual data patterns for performance validation",
    "Implement error handling for corrupted compressed data",
    "Monitor compression ratios to detect data pattern changes"
]

for practice in best_practices:
    print(f"  • {practice}")

print("File compression applications completed successfully!")

## Example 5: Advanced features and integration patterns

In [ ]:
import json
import sys
import time
import lz4.block

# Example 5: Advanced features and integration patterns
print("Example 5: Advanced Features and Integration Patterns")

# Structured, moderately repetitive records: the block modes trade speed for ratio on data like this
sample_data = b"".join(
    b"record %05d: user=user%03d status=%s value=%d\n" % (i, i % 97, [b"active", b"idle", b"error"][i % 3], (i * 7919) % 10007)
    for i in range(2000)
)

print(f"Testing advanced LZ4 features:")
print(f"Sample data size: {len(sample_data)} bytes")

# Block compression modes: `compression` (1-12) only applies in mode="high_compression",
# and `acceleration` only in mode="fast"
compression_modes = [
    ("Default", {"mode": "default"}),
    ("High compression 9", {"mode": "high_compression", "compression": 9}),
    ("Fast, accel. 8", {"mode": "fast", "acceleration": 8}),
]

print(f"\nCompression mode comparison:")
print(f"{'Mode':<20} {'Size':<8} {'Ratio':<6} {'Time (ms)':<10}")
print("-" * 48)

compressed_sizes = {}
for mode_name, params in compression_modes:
    start_time = time.perf_counter()
    compressed = lz4.block.compress(sample_data, **params)
    compress_time = time.perf_counter() - start_time
    compressed_sizes[mode_name] = len(compressed)
    
    ratio = len(sample_data) / len(compressed)
    print(f"{mode_name:<20} {len(compressed):<8} {ratio:<6.2f} {compress_time * 1000:<10.3f}")
    
    # Every mode produces a standard LZ4 block that decompresses the same way
    assert lz4.block.decompress(compressed) == sample_data, mode_name

assert compressed_sizes["High compression 9"] < compressed_sizes["Default"], compressed_sizes

# Integration patterns
print(f"\nIntegration patterns:")

# Pattern 1: Context manager for compression
class LZ4Compressor:
    def __init__(self):
        self.data_buffer = []
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        pass
    
    def add_data(self, data):
        self.data_buffer.append(data)
    
    def compress_all(self):
        combined = b"".join(self.data_buffer)
        return lz4.block.compress(combined)

# Demonstrate context manager
with LZ4Compressor() as compressor:
    compressor.add_data(b"First chunk of data")
    compressor.add_data(b"Second chunk of data")
    compressor.add_data(b"Third chunk of data")
    
    compressed_result = compressor.compress_all()

print(f"Context manager compression: {len(compressed_result)} bytes")

# Pattern 2: Utility functions
def compress_dict(data_dict):
    """Compress dictionary data to bytes"""
    json_str = json.dumps(data_dict).encode('utf-8')
    return lz4.block.compress(json_str)

def decompress_dict(compressed_data):
    """Decompress bytes back to dictionary"""
    json_bytes = lz4.block.decompress(compressed_data)
    return json.loads(json_bytes.decode('utf-8'))

# Test utility functions
test_dict = {"users": [f"user{i}" for i in range(100)], "active": True}
compressed_dict = compress_dict(test_dict)
recovered_dict = decompress_dict(compressed_dict)

print(f"Dictionary compression: {sys.getsizeof(test_dict)} -> {len(compressed_dict)} bytes")
assert recovered_dict == test_dict
assert lz4.block.decompress(compressed_result) == b"First chunk of dataSecond chunk of dataThird chunk of data"
print("Dictionary recovery: round trip matches")

# Performance characteristics
print(f"\nLZ4 characteristics:")
characteristics = [
    "Speed focus: Optimized for decompression speed over ratio",
    "Memory efficient: Low memory overhead during processing", 
    "Streaming capable: Support for streaming compression/decompression",
    "Cross-platform: Consistent behavior across operating systems",
    "Industry standard: Wide adoption in databases and storage systems"
]

for char in characteristics:
    print(f"  • {char}")

# When to use LZ4
print(f"\nWhen to use LZ4:")
use_cases = [
    "Real-time data processing where speed is critical",
    "Network protocols requiring low latency",
    "Database storage with frequent read access",
    "In-memory compression for space-constrained systems",
    "Backup systems where decompression speed matters"
]

for use_case in use_cases:
    print(f"  ✓ {use_case}")

print("Advanced features and integration completed successfully!")